# Dealing with Conflicts

This file shows how `Larkita` deals with *shift-reduce* and *reduce-reduce* conflicts.
The following grammar is *ambiguous* because it does not specify the precedence of the arithmetical operators:
```
    expr : expr "+" expr
         | expr "*" expr
         | NUMBER
```

## Specification of the Scanner and the Parser

The operators `"+"` and `"*"` occur as *string literals* inside the grammar rules.  `Larkita` uses the string
literals of an alternative as the *head* of the abstract syntax tree that is built for this alternative:
the alternative `expr "+" expr` yields the tuple `('+', lhs, rhs)` and a number yields a pair like `('NUMBER', '42')`.
(In OCaml, these tuples are values of the type `ast`, see `../Chapter-07/01-Calculator.ipynb`.)  How the abstract syntax tree is built has no influence on the parse table.

In [ ]:
#use "../lib/nb.ml";;
load_lib "larkita.ml";;
cat "conflicts.lark"

## How `Larkita` Reports Its Conflicts

`Larkita` always generates a *canonical LR(1) parser*.  It reports conflicts in two ways:

1. Shift-reduce conflicts are resolved in favour of shifting.  For every grammar with shift-reduce conflicts,
   `Larkita` issues a warning, which is printed below the cell that creates the parser.  (In Python, it is a
   warning of class `ConflictWarning`.)  Reduce-reduce conflicts are errors:  they raise the exception `Larkita.Grammar_error`.
2. `Larkita` always writes a description of all states of the LR(1) automaton into an *output file*.  Among other
   things, this file lists all conflicts.  The argument `~out_file` specifies the name of this file.

In contrast to `Lark`, whose *default* parser is an *Earley* parser that has no problem at all with an ambiguous
grammar, `Larkita` has no other parsing algorithm.  Hence the conflicts cannot go unnoticed.

In [ ]:
let parser = Larkita.open_ ~out_file:"conflicts.out" "conflicts.lark"

The section `Conflicts` of the output file lists the conflicts.  Every entry names

1. the state and the terminal for which the entry of the action table is ambiguous,
2. the action that has been chosen, and
3. the alternatives:  the state the parser could shift to and the grammar rule that could have been used to reduce
   the symbol stack in this situation.

Note the remark `resolved as shift`:  `Larkita` resolves **every** shift-reduce conflict in favour of shifting.

In [ ]:
(* report_index report sub is the position of the first occurrence of sub in report, like report.index(sub) *)
let report_index (report : string) (sub : string) : int =
  let n = String.length sub in
  let rec find i = if String.sub report i n = sub then i else find (i + 1) in
  find 0

(* the part of report between the strings start and stop, like report[report.index(start):report.index(stop)] *)
let report_slice (report : string) (start : string) (stop : string) : string =
  let i = report_index report start in
  String.sub report i (report_index report stop - i)

(* split s sep splits the string s at the occurrences of the string sep, like s.split(sep) *)
let split (s : string) (sep : string) : string list =
  let n = String.length sep in
  let rec go start i acc =
    if i + n > String.length s then List.rev (String.sub s start (String.length s - start) :: acc)
    else if String.sub s i n = sep then go (i + n) (i + n) (String.sub s start (i - start) :: acc)
    else go start (i + 1) acc
  in
  go 0 0 []

(* contains s sub checks whether sub occurs in s, like sub in s *)
let contains (s : string) (sub : string) : bool =
  let n = String.length sub in
  let rec go i = i + n <= String.length s && (String.sub s i n = sub || go (i + 1)) in
  go 0

In [ ]:
let report () : string = read_file "conflicts.out"
;;
print_string (report_slice (report ()) "Conflicts\n" "States\n");
flush stdout

The only thing we can do is to declare that we do not want to tolerate any conflicts at all.  The function
`Larkita.with_conflict_warnings` can turn the warning into the exception `Larkita.Conflict_warning`, like the
module `warnings` of `Python` turns the `ConflictWarning` into an exception.  It is a good idea to develop a grammar this way, for
then a conflict cannot be overlooked.

In [ ]:
Larkita.with_conflict_warnings `Raise (fun () ->
    try ignore (Larkita.open_ ~out_file:"conflicts.out" "conflicts.lark")
    with Larkita.Conflict_warning e -> print_endline e)

## Inspecting the LR(1) States

The section `States` of the output file describes all states of the canonical LR(1) automaton.  Every state lists
its *marked rules* together with their *lookahead* tokens, i.e. the tokens that may follow the rule.  The marked rules
that are added by the closure are marked with a `+`.  Below the marked rules, the actions and the gotos of the state
are listed.  The rules are numbered as in the section `Rules` at the beginning of the file.

Let us look at the states.  `Larkita` marks the position of the parser inside a rule with the bullet `•` that we use
in the lecture notes, and the end of the input is called `$END`.

In the states 5 and 6, which contain a marked rule of the form `expr → expr "+" expr •` or `expr → expr "*" expr •`,
there are two shift-reduce conflicts each.  All of them are resolved in favour of shifting: the only reduce action
that survives in these states is the one for `$END`.  The discarded reduce actions are listed below the actions of
these states.

In [ ]:
let report = report () in
let i = report_index report "States\n" in
print_string (String.sub report i (String.length report - i));
flush stdout

## Building an Abstract Syntax Tree

The parser generated by `Larkita` returns the abstract syntax tree as a nested tuple.  Hence we can use the function
`ast2dot` of the notebook `../AST2Dot.ipynb` directly in order to display it.

In [ ]:
run_notebook "../AST2Dot.ipynb"

The function `test(s)` takes a string `s` as its argument and tries to parse this string.  If all goes well,
the abstract syntax tree is returned and displayed.  If the string can't be parsed, `Larkita` raises an
exception `Larkita.Unexpected_input`, which we catch in order to print a readable error message.

In [ ]:
let test (s : string) : ast option =
  match Larkita.parse parser s with
  | t ->
      let d = ast2dot t in
      Graphviz.display d;
      Some t
  | exception Larkita.Unexpected_input e ->
      Printf.printf "Syntax error:\n%s\n%!" e.message;
      None

The next example shows that this parser does not produce the abstract syntax that reflects the precedences
of the arithmetical operators:  the string `2*3+4` is parsed as `2*(3+4)`.

In [ ]:
test "2*3+4"

Due to the fact that all shift-reduce conflicts are resolved in favour of shift, all operators have the
same precedence and effectively associate to the right.

In [ ]:
test "2+3+4"

In [ ]:
test "1+2*3"

The notebook [02-Conflicts-Resolved.ipynb](02-Conflicts-Resolved.ipynb) shows how these conflicts can
be removed.  Since `Larkita` does not support operator precedence declarations, the precedences and
associativities of the operators have to be encoded into the grammar by introducing one syntactical
variable per precedence level.